In [1]:
import json
import numpy as np
import pandas as pd
from sklearn.model_selection import StratifiedGroupKFold

In [2]:
N_FOLDS = 5
SEED = 42

UID = "UID"
PARENT = "parent_uid"
FORMULA = "Chemical"
SUCCESS = "Cat_Success"
IMPURITY = "Impurity_Percentage"
RATIO = "P2O3ratio"

MODELS = ("synth", "impurity", "p2o3")
A00_MODES = ("none", "add", "only")

DATA_PATH = "dataset_splits/featurised_clean.csv"
FOLDS_PATH = "dataset_splits/folds.csv"

In [3]:
def is_success(v):
    """Cat_Success may be bool, number or string."""
    if isinstance(v, str):
        return v.strip().lower() in {"successful", "success", "true", "1", "yes"}
    try:
        return bool(v) and not pd.isna(v)
    except TypeError:
        return False

def load_data(data_path="dataset_splits/featurised_clean.csv",
              folds_path="dataset_splits/folds.csv",
              feat_path="dataset_splits/feature_cols.json"):
    """Features + metadata joined with the saved fold column.

    Returns (df, feat_cols). Adds `is_u00` and `success` flags.
    """
    df = pd.read_csv(data_path)
    folds = pd.read_csv(folds_path)[[UID, "fold"]]
    df = df.merge(folds, on=UID, how="left", validate="one_to_one")
    if df["fold"].isna().any():
        missing = df.loc[df["fold"].isna(), UID].tolist()
        raise ValueError(f"{len(missing)} rows have no fold (e.g. {missing[:5]}) - "
                         "rerun the folds notebook after changing the dataset")
    df["fold"] = df["fold"].astype(int)
    df["is_u00"] = df[UID].astype(str).str.startswith("U00")
    df["success"] = df[SUCCESS].apply(is_success)

    with open(feat_path) as f:
        feat_cols = json.load(f)
    return df, feat_cols

def _eligible(df, model):
    """Rows usable for this model's target (any fold, U00 and A00)."""
    if model == "synth":
        return df[df["is_u00"]]                                 # A00 always successful
    if model == "impurity":
        return df[df["success"] & df[IMPURITY].notna()]
    if model == "p2o3":
        r = df[RATIO]
        return df[df["success"] & r.notna() & (r >= 0)]          # drops -1 / -2 codes
    raise ValueError(f"model must be one of {MODELS}")

def get_split(df, k, model, a00="none"):
    """Train / test rows for fold k.

    model : "synth" | "impurity" | "p2o3"
    a00   : "none" - train on U00 only
            "add"  - train on U00 + A00 (A00 whose parent is in fold k excluded)
            "only" - train on A00 only
            (A00 modes only apply to "p2o3"; impurity uses A00 as a check instead,
             see a00_check_rows)
    """
    if a00 not in A00_MODES:
        raise ValueError(f"a00 must be one of {A00_MODES}")
    if a00 != "none" and model != "p2o3":
        raise ValueError("A00 training rows are only used for model='p2o3'")

    rows = _eligible(df, model)
    u00, a00_rows = rows[rows["is_u00"]], rows[~rows["is_u00"]]

    test = u00[u00["fold"] == k]
    train_u00 = u00[u00["fold"] != k]
    train_a00 = a00_rows[a00_rows["fold"] != k]                 # parent not in test

    train = {"none": train_u00,
             "add": pd.concat([train_u00, train_a00]),
             "only": train_a00}[a00]

    _assert_no_leak(train, test)
    return train, test

def a00_check_rows(df, k):
    """A00 rows whose parent is in test fold k.

    Predicting these with the fold-k model gives out-of-fold predictions for
    both the parent and its adjusted composition (impurity sanity check).
    """
    return df[~df["is_u00"] & (df["fold"] == k)]

def _assert_no_leak(train, test):
    assert test["is_u00"].all(), "test set must be U00 only"
    shared = set(train.loc[train["is_u00"], FORMULA]) & set(test[FORMULA])
    assert not shared, f"composition in both train and test: {sorted(shared)[:5]}"
    parents_in_test = set(train.loc[~train["is_u00"], PARENT]) & set(test[UID])
    assert not parents_in_test, f"A00 trained while parent tested: {sorted(parents_in_test)[:5]}"

def split_summary(df):
    """Rows per fold for every model / A00 mode - quick sanity table."""
    out = []
    for k in range(N_FOLDS):
        row = {"fold": k}
        for model in MODELS:
            train, test = get_split(df, k, model)
            row[f"{model}_train"], row[f"{model}_test"] = len(train), len(test)
        row["p2o3_train_add"] = len(get_split(df, k, "p2o3", "add")[0])
        row["p2o3_train_only"] = len(get_split(df, k, "p2o3", "only")[0])
        row["a00_check"] = len(a00_check_rows(df, k))
        out.append(row)
    return pd.DataFrame(out).set_index("fold")

In [4]:
df = pd.read_csv(DATA_PATH)
df["is_u00"] = df[UID].astype(str).str.startswith("U00")
df["success"] = df[SUCCESS].apply(is_success)

u00 = df[df["is_u00"]]
a00 = df[~df["is_u00"]]
print(f"U00: {len(u00)} rows, {u00[FORMULA].nunique()} compositions | A00: {len(a00)} rows")

U00: 369 rows, 123 compositions | A00: 188 rows


/tmp/ipykernel_223206/2762783019.py:2: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["is_u00"] = df[UID].astype(str).str.startswith("U00")
/tmp/ipykernel_223206/2762783019.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["success"] = df[SUCCESS].apply(is_success)


In [5]:
sgkf = StratifiedGroupKFold(n_splits=N_FOLDS, shuffle=True, random_state=SEED)

fold_of = pd.Series(-1, index=u00.index)
for k, (_, test_idx) in enumerate(sgkf.split(u00, u00["success"], groups=u00[FORMULA])):
    fold_of.iloc[test_idx] = k

df["fold"] = -1
df.loc[u00.index, "fold"] = fold_of

/tmp/ipykernel_223206/3447713801.py:7: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["fold"] = -1


A00 rows inherit their parent's fold

In [6]:
parent_fold = df.loc[df["is_u00"]].set_index(UID)["fold"]
df.loc[~df["is_u00"], "fold"] = df.loc[~df["is_u00"], PARENT].map(parent_fold)

assert df["fold"].notna().all() and (df["fold"] >= 0).all(), "some rows have no fold"
df["fold"] = df["fold"].astype(int)

In [7]:
# every composition sits in exactly one fold
folds_per_comp = df[df["is_u00"]].groupby(FORMULA)["fold"].nunique()
assert (folds_per_comp == 1).all(), "a composition is split across folds"

# every A00 is in its parent's fold
a = df[~df["is_u00"]]
assert (a["fold"].values == a[PARENT].map(parent_fold).values).all()

u = df[df["is_u00"]]
ratio_ok = u["success"] & u[RATIO].notna() & (u[RATIO] >= 0)
summary = pd.DataFrame({
    "U00_rows": u.groupby("fold").size(),
    "compositions": u.groupby("fold")[FORMULA].nunique(),
    "success_rate": u.groupby("fold")["success"].mean().round(3),
    "successful": u[u["success"]].groupby("fold").size(),
    "p2o3_usable": u[ratio_ok].groupby("fold").size(),
    "A00_rows": a.groupby("fold").size(),
}).fillna(0).astype({"A00_rows": int})
summary

,U00_rows,compositions,success_rate,successful,p2o3_usable,A00_rows
fold,,,,,,
0,75,25,0.667,50,50,39
1,75,25,0.667,50,50,40
2,72,24,0.653,47,47,35
3,75,25,0.653,49,49,36
4,72,24,0.667,48,48,38


In [8]:
df[[UID, PARENT, FORMULA, "is_u00", "fold"]].to_csv(FOLDS_PATH, index=False)
print(f"Saved {len(df)} rows to {FOLDS_PATH}")

Saved 557 rows to dataset_splits/folds.csv


Reload through the shared loader and confirm every model's split passes the
leakage checks 

In [9]:
df_check, feat_cols = load_data()
split_summary(df_check)

/tmp/ipykernel_223206/2788646511.py:25: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["is_u00"] = df[UID].astype(str).str.startswith("U00")
/tmp/ipykernel_223206/2788646511.py:26: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  df["success"] = df[SUCCESS].apply(is_success)


,synth_train,synth_test,impurity_train,impurity_test,p2o3_train,p2o3_test,p2o3_train_add,p2o3_train_only,a00_check
fold,,,,,,,,,
0,294,75,194,50,194,50,343,149,39
1,294,75,194,50,194,50,342,148,40
2,297,72,197,47,197,47,350,153,35
3,294,75,195,49,195,49,347,152,36
4,297,72,196,48,196,48,346,150,38
